In [1]:
# 1. 数据准备：读取类别 + 动态生成 8000 张抽样清单
import json
import random
import yaml
from pathlib import Path

# 原始挂载路径
DATA_ROOT = Path("/kaggle/input/datasets/shilljinglza/traffic-data/TrafficDetYolo")
DATA_YAML = str(DATA_ROOT / "data.yaml")

# 读取类别并保存
names = yaml.safe_load(open(DATA_YAML, encoding="utf-8"))["names"]
json.dump(names, open("/kaggle/working/traffic_classes.json", "w"), ensure_ascii=False, indent=2)

print("✅ 数据就绪:", DATA_ROOT)
print("类别列表:", names)

# 从 train 图片中随机抽取 8000 张
train_img_dir = DATA_ROOT / "train" / "images"
all_imgs = [f.name for f in train_img_dir.iterdir() if f.suffix.lower() in ['.jpg', '.jpeg', '.png']]

random.seed(42)  # 固定种子，保证结果可复现
sample_imgs = random.sample(all_imgs, min(8000, len(all_imgs)))

# 把抽样出来的图片路径写入一个 txt 清单
sample_txt_path = "/kaggle/working/train_sample_v11.txt"
with open(sample_txt_path, "w") as f:
    for img in sample_imgs:
        f.write(f"{DATA_ROOT}/train/images/{img}\n")

data = yaml.safe_load(open(DATA_YAML, encoding="utf-8"))
data["path"] = str(DATA_ROOT)
data["train"] = sample_txt_path  # 指向刚才生成的清单
data["val"] = "val/images"

SAMPLE_YAML = "/kaggle/working/traffic_sample_v11.yaml"
with open(SAMPLE_YAML, "w") as f:
    yaml.dump(data, f, sort_keys=False)

print(f"\n✅ 抽样配置生成完毕！")
print(f"本次训练图片数: {len(sample_imgs)} 张")
print(f"后续训练请使用该配置: {SAMPLE_YAML}")

✅ 数据就绪: /kaggle/input/datasets/shilljinglza/traffic-data/TrafficDetYolo
类别列表: ['Vehicle', 'Bus', 'Bicycle', 'Person', 'Engine', 'Truck', 'Tricycle', 'Obstacle', 'Pothole', 'Traffic Light', 'Traffic Sign']

✅ 抽样配置生成完毕！
本次训练图片数: 8000 张
后续训练请使用该配置: /kaggle/working/traffic_sample_v11.yaml


In [2]:
# 2. 安装 ultralytics + 预下载 yolo11s.pt（注意权重名是 yolo11s，不是 yolov11s）
import importlib.util
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec("ultralytics") is None:
    print("正在安装 ultralytics...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)

from ultralytics import YOLO
# YOLO('yolo11s.pt') 会自动从 GitHub 下载权重
model = YOLO("yolo11s.pt")
print("✅ 环境准备完毕，yolo11s.pt 已就绪")

正在安装 ultralytics...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 3.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
✅ 环境准备完毕，yolo11s.pt 已就绪


In [ ]:
# 3. 实验配置与训练（导包直接调用）
from ultralytics import YOLO

EPOCHS, IMGSZ, BATCH, SEED = 5, 320, 4, 42
PROJECT_DIR = "/kaggle/working/traffic_models_v11"

RUNS = [
    dict(name="v11_baseline_lr0.01_wd0.0005", lr=0.01,  wd=0.0005),
    dict(name="v11_lr0.005",                  lr=0.005, wd=0.0005),
    dict(name="v11_lr0.001",                  lr=0.001, wd=0.0005),
]
BEST_LR = 0.01
for _wd in (0.001, 0.0001):
    RUNS.append(dict(name=f"v11_wd{_wd}", lr=BEST_LR, wd=_wd))

for run in RUNS:
    print(f"\n{'='*60}")
    print(f"🚀 开始训练: {run['name']} (lr={run['lr']}, wd={run['wd']})")
    print(f"{'='*60}\n")

    model = YOLO("yolo11s.pt")
    model.train(
        data="/kaggle/working/traffic_sample_v11.yaml",
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        seed=SEED,
        optimizer='SGD',
        lr0=run['lr'],
        lrf=1.0,                 # 关闭余弦衰减，保证与YOLOv8公平对比
        weight_decay=run['wd'],
        project=PROJECT_DIR,
        name=run['name'],
        exist_ok=True,
        device=0,
        workers=2,
        cache=False,
        verbose=False
    )

print("\n✅ 全部 YOLOv11s 消融实验训练完成！")


🚀 开始训练: v11_baseline_lr0.01_wd0.0005 (lr=0.01, wd=0.0005)

Ultralytics 8.4.165 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/traffic_sample_v11.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=1.0, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0

In [ ]:
# 4. 汇总各次实验指标
import pandas as pd
import json
from pathlib import Path

RUN_DIR = Path("/kaggle/working/traffic_models_v11")
RUNS = [
    dict(name="v11_baseline_lr0.01_wd0.0005", lr=0.01,  wd=0.0005),
    dict(name="v11_lr0.005",                  lr=0.005, wd=0.0005),
    dict(name="v11_lr0.001",                  lr=0.001, wd=0.0005),
    dict(name="v11_wd0.001",                  lr=0.01,  wd=0.001),
    dict(name="v11_wd0.0001",                 lr=0.01,  wd=0.0001),
]

def pick_col(df, *needles):
    for c in df.columns:
        if all(k in c for k in needles):
            return c
    raise KeyError(f"未找到包含 {needles} 的列")

def parse_run(name, lr, wd):
    csv_path = RUN_DIR / name / "results.csv"
    if not csv_path.exists():
        print(f"⚠️ 跳过 {name}: 未找到结果文件")
        return None
        
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    if "epoch" not in df.columns:
        df.insert(0, "epoch", range(len(df)))
        
    try:
        c_ep = pick_col(df, "epoch")
        c_p = pick_col(df, "precision")
        c_r = pick_col(df, "recall")
        c_m50 = pick_col(df, "mAP50")
        c_m5095 = pick_col(df, "mAP50-95")
    except KeyError as e:
        print(f"⚠️ 解析 {name} 失败: {e}")
        return None

    best_i = df[c_m50].idxmax()
    best = df.loc[best_i]
    
    return dict(name=name, lr=lr, wd=wd, epochs=int(best[c_ep]) + 1,
                best_epoch=int(best[c_ep]) + 1,
                precision=float(best[c_p]),
                recall=float(best[c_r]),
                mAP50=float(best[c_m50]),
                mAP5095=float(best[c_m5095]),
                curves=dict(epoch=df[c_ep].astype(int).tolist(),
                            mAP50=df[c_m50].tolist(),
                            mAP5095=df[c_m5095].tolist()))

summaries = [s for s in (parse_run(**r) for r in RUNS) if s is not None]

if summaries:
    tbl = pd.DataFrame(summaries)[["name", "lr", "wd", "precision", "recall", "mAP50", "mAP5095"]]
    print("📊 YOLOv11s 消融实验结果汇总（按 mAP50 降序排列）")
    print("="*60)
    print(tbl.sort_values("mAP50", ascending=False).to_string(index=False))
    
    json.dump(summaries, open("/kaggle/working/yolov11_runs_summary.json", "w"), ensure_ascii=False, indent=2)
    
    best = max(summaries, key=lambda s: s["mAP50"])
    print(f"\n🏆 表现最好的参数组合: {best['name']}")
    print(f"👉 最优 lr = {best['lr']}, 最优 wd = {best['wd']}, 最高 mAP50 = {best['mAP50']:.4f}")
else:
    print("❌ 没有找到任何有效实验结果")

In [ ]:
# 5. 选出最优实验并导出权重
import shutil, pathlib

best = max(summaries, key=lambda s: s["mAP50"])
src = pathlib.Path("/kaggle/working/traffic_models_v11") / best["name"] / "weights/best.pt"
shutil.copy2(src, "/kaggle/working/yolo11s_traffic_best.pt")
print(f"✅ 已导出测试权重: {best['name']} (mAP50={best['mAP50']:.4f})")

In [ ]:
# 6. 绘制消融对比曲线
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))
for s in summaries:
    plt.plot(s["curves"]["epoch"], s["curves"]["mAP50"], marker="o", label=s["name"])
plt.xlabel("epoch"); plt.ylabel("mAP@0.5"); plt.grid(alpha=.3); plt.legend(fontsize=8)
plt.title("YOLOv11s ablation: mAP@0.5")
plt.tight_layout()
plt.savefig("/kaggle/working/yolov11_ablation_curves.png", dpi=150)
plt.show()

In [ ]:
# 7. 生成全量训练的绝对路径 yaml
import yaml
from pathlib import Path

DATA_ROOT = "/kaggle/input/datasets/shilljinglza/traffic-data/TrafficDetYolo"
with open(Path(DATA_ROOT) / "data.yaml", "r", encoding="utf-8") as f:
    data = yaml.safe_load(f)
data["path"] = DATA_ROOT
data["train"] = "train/images"
data["val"] = "val/images"

NEW_YAML = "/kaggle/working/traffic_data_abs_v11.yaml"
with open(NEW_YAML, "w", encoding="utf-8") as f:
    yaml.dump(data, f, sort_keys=False)
print(f"✅ 全量训练 yaml 已生成: {NEW_YAML}")

In [ ]:
# 8. 全量数据最终训练（记得把 BEST_LR 和 BEST_WD 替换成你刚刚跑出的最优值）
from ultralytics import YOLO

# # ⚠️ 替换成第4步找出的最优参数
# BEST_LR = 0.001    # 根据你的实际结果修改
# BEST_WD = 0.0005   # 根据你的实际结果修改

model = YOLO("yolo11s.pt")
model.train(
    data="/kaggle/working/traffic_data_abs_v11.yaml",
    imgsz=640,
    epochs=20,
    patience=10,
    optimizer='SGD',
    lr0=BEST_LR,
    lrf=0.1,
    weight_decay=BEST_WD,
    batch=8,
    seed=42,
    project="/kaggle/working/traffic_models_v11_full",
    name="yolo11s_full_best",
    exist_ok=True,
    device=0,
    workers=2,
    cache=False
)

In [ ]:
# 9. 打包全量训练结果
import shutil
import os
from pathlib import Path

RESULT_DIR = Path("/kaggle/working/traffic_models_v11_full/yolo11s_full_best")
ZIP_PATH = "/kaggle/working/yolo11s_full_best_results.zip"

if not RESULT_DIR.exists():
    print(f"❌ 找不到结果目录: {RESULT_DIR}")
else:
    if os.path.exists(ZIP_PATH):
        os.remove(ZIP_PATH)
    print(f"正在打包: {RESULT_DIR} ...")
    shutil.make_archive(base_name=ZIP_PATH.replace(".zip", ""), format='zip', root_dir=RESULT_DIR)
    zip_size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
    print(f"\n✅ 打包完成！")
    print(f"压缩包路径: {ZIP_PATH}")
    print(f"压缩包大小: {zip_size_mb:.1f} MB")
    print("\n📦 压缩包内包含：")
    print("  - weights/best.pt  (最佳模型权重)")
    print("  - weights/last.pt  (最后一个epoch的权重)")
    print("  - results.csv      (训练过程数据)")
    print("  - confusion_matrix.png  (混淆矩阵)")
    print("  - F1_curve.png, PR_curve.png 等评估图表")